# MLP em PyTorch com Loop de Treinamento, Early Stopping e Batching

## Objetivo
Implementar um **loop de treinamento explícito** com **validação**, **early stopping** (paciência sobre loss de validação) e **batching** organizado via `DataLoader`. Este notebook estende o trabalho do notebook `03_mlp_pytorch.ipynb` com práticas mais realistas e reprodutíveis.

### O que será feito (passo a passo)
1. Carregar e preparar dados com o mesmo padrão dos baselines (`Telco_customer_churn_ready.csv`).
2. Dividir em **treino**, **validação** e **teste** (ou treino+validação vs teste).
3. Padronizar features com `StandardScaler` apenas no treino (sem vazamento).
4. Criar `DataLoader` para batching reproduzível.
5. Definir MLP com `torch.nn.Module`.
6. Implementar loop de treinamento com:
   - Iteração sobre epochs com mini-batches.
   - Cálculo de loss e atualização de pesos.
   - Avaliação em validação a cada epoch.
   - **Early stopping** com paciência sobre loss de validação.
7. Avaliar no conjunto de teste com métricas significativas.
8. Registrar experimento no MLflow (parâmetros, métricas, modelo).

---

### Por que Early Stopping?
Modelos neurais podem sofrer **overfitting**: a loss de treinamento continua diminuindo enquanto a de validação começa a aumentar. O **early stopping** monitora a validação e para o treino quando percebe que o modelo não está mais generalizando bem, economizando tempo e melhorando performance no teste.

### Por que Batching?
Treinar com mini-batches é mais eficiente, oferece regularização estatística e é padrão em produção. PyTorch's `DataLoader` organiza isso de forma limpa.

## Importações e Configuração

In [ ]:
from pathlib import Path
from copy import deepcopy

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset
import mlflow
import mlflow.pytorch

# Configuração de seeds para reprodutibilidade.
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED) if torch.cuda.is_available() else None

# Configuração de PyTorch para reproducibilidade (pode desacelerar um pouco).
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Verificar se GPU está disponível (opcional).
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {DEVICE}")

# Procurar o arquivo de dados tratado (mesmo padrão do notebook 02).
READY_PATH_CANDIDATES = [
    Path("../data/Telco_customer_churn_ready.csv"),
    Path("data/Telco_customer_churn_ready.csv"),
]
DATA_PATH = next((p for p in READY_PATH_CANDIDATES if p.exists()), None)

assert DATA_PATH is not None, (
    "Arquivo tratado do EDA não encontrado. Rode no notebook de EDA para gerar Telco_customer_churn_ready.csv."
)
print(f"Arquivo de entrada encontrado: {DATA_PATH.resolve()}")

## Carregamento e Pré-processamento dos Dados

In [ ]:
# Carrega o dataset já tratado no EDA.
TARGET = "Churn Value"
df = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {df.shape}")
print(f"Colunas: {df.columns.tolist()[:10]}...")  # Mostra primeiras colunas.

# Verifica a coluna alvo.
assert TARGET in df.columns, f"Coluna alvo '{TARGET}' não encontrada."
print(f"\nDistribuição do alvo:\n{df[TARGET].value_counts(normalize=True)}")

In [ ]:
# Separa features (X) e alvo (y).
X = df.drop(columns=[TARGET])
y = df[TARGET].astype(int)

# Converte para numpy (float32 para compatibilidade com PyTorch).
X_np = X.astype(np.float32).to_numpy()
y_np = y.to_numpy(dtype=np.float32)

# Split treino/teste (80/20 com estratificação).
# Depois, o treino será dividido em treino/validação (ex: 75% treino, 25% validação do treino original).
X_train_temp, X_test, y_train_temp, y_test = train_test_split(
    X_np,
    y_np,
    test_size=0.2,
    random_state=SEED,
    stratify=y_np,
)

# Split treino/validação (75% treino, 25% validação do treino).
X_train, X_val, y_train, y_val = train_test_split(
    X_train_temp,
    y_train_temp,
    test_size=0.25,
    random_state=SEED,
    stratify=y_train_temp,
)

print(f"Treino: {len(X_train)} | Validação: {len(X_val)} | Teste: {len(X_test)}")
print(f"Feature shape: {X_train.shape[1]} features")

In [ ]:
# Padroniza features com StandardScaler (FIT apenas no treino).
# Isto evita "vazamento de informação" do teste para o treino.
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train).astype(np.float32)
X_val_s = scaler.transform(X_val).astype(np.float32)
X_test_s = scaler.transform(X_test).astype(np.float32)

print("Padronização concluída (sem vazamento).")
print(f"Média treino (amostra): {X_train_s[:5, 0]}")
print(f"Média validação (amostra): {X_val_s[:5, 0]}")

## Criação de DataLoaders

In [ ]:
# Hiperparâmetros de treinamento.
BATCH_SIZE = 64
LR = 0.001  # Learning Rate.
NUM_EPOCHS = 100  # Máximo de epochs (pode parar antes com early stopping).
PATIENCE = 10  # Paciência para early stopping (quantos epochs sem melhoria antes de parar).

# Cria TensorDataset para cada split.
train_ds = TensorDataset(
    torch.from_numpy(X_train_s),
    torch.from_numpy(y_train).unsqueeze(1),  # Transforma em coluna (n_samples, 1).
)
val_ds = TensorDataset(
    torch.from_numpy(X_val_s),
    torch.from_numpy(y_val).unsqueeze(1),
)
test_ds = TensorDataset(
    torch.from_numpy(X_test_s),
    torch.from_numpy(y_test).unsqueeze(1),
)

# Cria DataLoaders com shuffle no treino (melhora generalização).
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, random_state=SEED)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}")

## Definição da Arquitetura MLP

In [ ]:
class MLPChurn(nn.Module):
    """
    MLP simples para classificação binária de churn.
    
    Arquitetura:
    - Camada de entrada: input_dim features
    - Camada oculta 1: 64 neurônios com ReLU
    - Camada oculta 2: 32 neurônios com ReLU
    - Camada de saída: 1 neurônio (logit, sem ativação — usaremos BCEWithLogitsLoss)
    """
    
    def __init__(self, input_dim, hidden_dim_1=64, hidden_dim_2=32):
        super(MLPChurn, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim_1)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim_1, hidden_dim_2)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(hidden_dim_2, 1)
        
    def forward(self, x):
        """Forward pass."""
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.fc3(x)
        return x

# Instancia o modelo.
n_features = X_train_s.shape[1]
model = MLPChurn(input_dim=n_features).to(DEVICE)
print(f"Modelo instanciado no dispositivo: {DEVICE}")
print(model)

## Definição de Loss e Otimizador

In [ ]:
# Loss function: BCEWithLogitsLoss é numericamente estável para classificação binária.
# Combina sigmoid + BCE, evitando instabilidade numérica.
criterion = nn.BCEWithLogitsLoss()

# Otimizador: Adam é simples e funciona bem em muitos casos.
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

print(f"Loss function: {criterion}")
print(f"Optimizer: {optimizer}")

## Loop de Treinamento com Early Stopping

In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    """
    Treina o modelo por um epoch completo.
    
    Args:
        model: instância do modelo.
        dataloader: DataLoader com dados de treino.
        criterion: função de loss.
        optimizer: otimizador (ex: Adam).
        device: 'cuda' ou 'cpu'.
    
    Returns:
        loss_meio: loss médio do epoch.
    """
    model.train()  # Coloca modelo em modo treino.
    total_loss = 0.0
    
    for X_batch, y_batch in dataloader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        
        # Forward pass.
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        
        # Backward pass.
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * X_batch.size(0)
    
    avg_loss = total_loss / len(dataloader.dataset)
    return avg_loss


def evaluate(model, dataloader, criterion, device):
    """
    Avalia o modelo em um conjunto de dados.
    
    Args:
        model: instância do modelo.
        dataloader: DataLoader com dados (validação ou teste).
        criterion: função de loss.
        device: 'cuda' ou 'cpu'.
    
    Returns:
        avg_loss: loss médio.
        y_true: labels verdadeiros (numpy).
        y_pred_proba: probabilidades preditas (numpy).
    """
    model.eval()  # Coloca modelo em modo avaliação (desativa dropout, etc).
    total_loss = 0.0
    y_true_list = []
    y_pred_proba_list = []
    
    with torch.no_grad():  # Desativa cálculo de gradientes (mais rápido e menos memória).
        for X_batch, y_batch in dataloader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            total_loss += loss.item() * X_batch.size(0)
            
            # Converte logits para probabilidades com sigmoid.
            proba = torch.sigmoid(logits).cpu().numpy()
            y_true_list.append(y_batch.cpu().numpy())
            y_pred_proba_list.append(proba)
    
    avg_loss = total_loss / len(dataloader.dataset)
    y_true = np.vstack(y_true_list).flatten()
    y_pred_proba = np.vstack(y_pred_proba_list).flatten()
    
    return avg_loss, y_true, y_pred_proba

print("Funções de treino e avaliação definidas.")

In [ ]:
# Histórico de treino (para visualização).
history = {
    "train_loss": [],
    "val_loss": [],
}

# Early stopping: monitora o loss de validação.
best_val_loss = float("inf")
patience_counter = 0
best_model_state = None

print(f"Iniciando treinamento com early stopping (paciência={PATIENCE}, max_epochs={NUM_EPOCHS})...\n")

for epoch in range(NUM_EPOCHS):
    # Treina por um epoch.
    train_loss = train_epoch(model, train_loader, criterion, optimizer, DEVICE)
    
    # Avalia em validação.
    val_loss, _, _ = evaluate(model, val_loader, criterion, DEVICE)
    
    # Registra histórico.
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    
    # Verifica se validação melhorou.
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        # Salva o melhor estado do modelo.
        best_model_state = deepcopy(model.state_dict())
        print(f"Epoch {epoch+1:3d} | Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f} | [Melhor]")
    else:
        patience_counter += 1
        if (epoch + 1) % 5 == 0 or epoch == 0:  # Imprime a cada 5 epochs ou no primeiro.
            print(f"Epoch {epoch+1:3d} | Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f} | Paciência: {patience_counter}/{PATIENCE}")
    
    # Early stopping: para se paciência esgotou.
    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping! Paciência esgotada no epoch {epoch+1}.")
        break

# Carrega o melhor modelo encontrado.
if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print(f"\nMelhor modelo carregado (val_loss={best_val_loss:.6f}).")

## Avaliação no Conjunto de Teste

In [ ]:
# Avalia no teste com o melhor modelo.
test_loss, y_test_true, y_test_proba = evaluate(model, test_loader, criterion, DEVICE)

# Converte probabilidades em predições (threshold = 0.5).
THRESHOLD = 0.5
y_test_pred = (y_test_proba >= THRESHOLD).astype(int)

# Calcula métricas (conforme METRICAS.md).
accuracy = accuracy_score(y_test_true, y_test_pred)
precision = precision_score(y_test_true, y_test_pred, zero_division=0)
recall = recall_score(y_test_true, y_test_pred, zero_division=0)
f1 = f1_score(y_test_true, y_test_pred, zero_division=0)
roc_auc = roc_auc_score(y_test_true, y_test_proba)

print("="*60)
print("AVALIAÇÃO NO CONJUNTO DE TESTE")
print("="*60)
print(f"Test Loss: {test_loss:.6f}")
print(f"\nMétricas (threshold={THRESHOLD}):")
print(f"  Acurácia:  {accuracy:.4f}")
print(f"  Precisão: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1-Score:  {f1:.4f}")
print(f"  ROC-AUC:   {roc_auc:.4f}")
print("="*60)

## Visualização do Progresso de Treinamento

In [ ]:
import matplotlib.pyplot as plt

# Plot: Loss ao longo do treinamento.
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(history["train_loss"], label="Train Loss", marker="o", linestyle="-", markersize=3, alpha=0.7)
ax.plot(history["val_loss"], label="Val Loss", marker="s", linestyle="-", markersize=3, alpha=0.7)
ax.axvline(x=len(history["train_loss"])-1, color="red", linestyle="--", alpha=0.5, label="Early Stopping")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Evolução da Loss Durante o Treinamento")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Treinamento durou {len(history['train_loss'])} epochs.")

## Matriz de Confusão

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

# Calcula matriz de confusão.
cm = confusion_matrix(y_test_true, y_test_pred)

# Plot: Matriz de confusão.
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax, cbar=True,
            xticklabels=["Não Churn", "Churn"],
            yticklabels=["Não Churn", "Churn"])
ax.set_xlabel("Predito")
ax.set_ylabel("Verdadeiro")
ax.set_title("Matriz de Confusão (Teste)")
plt.tight_layout()
plt.show()

# Extrai componentes da matriz.
tn, fp, fn, tp = cm.ravel()
print(f"\nComponentes da Matriz de Confusão:")
print(f"  TN (Verdadeiro Negativo): {tn}")
print(f"  FP (Falso Positivo):      {fp}")
print(f"  FN (Falso Negativo):      {fn}")
print(f"  TP (Verdadeiro Positivo): {tp}")

## Integração com MLflow

In [ ]:
# Inicia um novo run no MLflow.
mlflow.set_experiment("MLP-Churn")

with mlflow.start_run(run_name="04_mlp_early_stopping"):
    # Registra parâmetros.
    mlflow.log_param("seed", SEED)
    mlflow.log_param("batch_size", BATCH_SIZE)
    mlflow.log_param("learning_rate", LR)
    mlflow.log_param("max_epochs", NUM_EPOCHS)
    mlflow.log_param("patience_early_stopping", PATIENCE)
    mlflow.log_param("threshold_prediction", THRESHOLD)
    mlflow.log_param("hidden_dim_1", 64)
    mlflow.log_param("hidden_dim_2", 32)
    mlflow.log_param("loss_function", "BCEWithLogitsLoss")
    mlflow.log_param("optimizer", "Adam")
    mlflow.log_param("input_features", n_features)
    
    # Registra métricas finais.
    mlflow.log_metric("test_loss", test_loss)
    mlflow.log_metric("test_accuracy", accuracy)
    mlflow.log_metric("test_precision", precision)
    mlflow.log_metric("test_recall", recall)
    mlflow.log_metric("test_f1_score", f1)
    mlflow.log_metric("test_roc_auc", roc_auc)
    mlflow.log_metric("best_val_loss", best_val_loss)
    mlflow.log_metric("epochs_trained", len(history["train_loss"]))
    
    # Registra o modelo.
    mlflow.pytorch.log_model(model, "model")
    
    print("\nExperimento registrado no MLflow!")
    print(f"ID do Run: {mlflow.active_run().info.run_id}")
    print(f"Experimento: {mlflow.active_run().info.experiment_id}")

## Resumo Final

In [ ]:
print("\n" + "="*80)
print("RESUMO DO EXPERIMENTO: MLP com Early Stopping e Batching")
print("="*80)
print(f"\nDataset: {DATA_PATH}")
print(f"Seed: {SEED}")
print(f"Splits: Treino={len(X_train)} | Val={len(X_val)} | Teste={len(X_test)}")
print(f"\nArquitetura MLP:")
print(f"  Input → 64 (ReLU) → 32 (ReLU) → 1 (Output)")
print(f"\nHiperparâmetros:")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Learning Rate: {LR}")
print(f"  Early Stopping Patience: {PATIENCE}")
print(f"  Epochs Treinados: {len(history['train_loss'])}")
print(f"\nResultados no Teste (threshold={THRESHOLD}):")
print(f"  Loss: {test_loss:.6f}")
print(f"  Acurácia: {accuracy:.4f}")
print(f"  Precisão: {precision:.4f}")
print(f"  Recall: {recall:.4f}")
print(f"  F1-Score: {f1:.4f}")
print(f"  ROC-AUC: {roc_auc:.4f}")
print(f"\nMatriz de Confusão (Teste):")
print(f"  TN={tn}, FP={fp}, FN={fn}, TP={tp}")
print("\nPróximas etapas:")
print("  - Comparar com baselines (05_compare_mlp_baselines.ipynb)")
print("  - Analisar trade-offs (06_tradeoff_custo_fp_fn.ipynb)")
print("  - Consolidar no MLflow (07_mlflow_experimentos_mlp_ensembles.ipynb)")
print("="*80)